# NMF on ModulAir 00684

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00684
data = pd.read_csv('/content/MOD-00684.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:42Z,577611581,2025-12-31T18:59:42Z,MOD-00684,49.4,0.1,11.578,0.908,0.229,0.048,0.052,...,33.422,25.941,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,4.20
2025-12-31T23:58:42Z,577611580,2025-12-31T18:58:42Z,MOD-00684,49.5,0.1,11.323,0.940,0.248,0.054,0.089,...,33.418,25.930,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,4.09
2025-12-31T23:57:42Z,577611578,2025-12-31T18:57:42Z,MOD-00684,49.8,0.1,12.160,1.016,0.181,0.042,0.057,...,34.116,25.193,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,5.70
2025-12-31T23:56:42Z,577609552,2025-12-31T18:56:42Z,MOD-00684,50.1,0.1,12.582,1.044,0.287,0.041,0.042,...,34.814,23.749,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,2.59
2025-12-31T23:55:42Z,577609550,2025-12-31T18:55:42Z,MOD-00684,49.8,0.1,11.556,0.975,0.286,0.045,0.041,...,33.879,25.193,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,1.91


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:42Z,2025-12-31T18:59:42Z,828.982,2.560,33.422,25.941,11.578,0.908,0.229,0.048,0.052,0.018
2025-12-31T23:58:42Z,2025-12-31T18:58:42Z,875.709,2.768,33.418,25.930,11.323,0.940,0.248,0.054,0.089,0.018
2025-12-31T23:57:42Z,2025-12-31T18:57:42Z,1005.216,2.897,34.116,25.193,12.160,1.016,0.181,0.042,0.057,0.037
2025-12-31T23:56:42Z,2025-12-31T18:56:42Z,909.765,2.896,34.814,23.749,12.582,1.044,0.287,0.041,0.042,0.028
2025-12-31T23:55:42Z,2025-12-31T18:55:42Z,817.203,2.897,33.879,25.193,11.556,0.975,0.286,0.045,0.041,0.009


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:42Z,828.982,2.560,33.422,25.941,11.578,0.908,0.229,0.048,0.052,0.018
1,2025-12-31T18:58:42Z,875.709,2.768,33.418,25.930,11.323,0.940,0.248,0.054,0.089,0.018
2,2025-12-31T18:57:42Z,1005.216,2.897,34.116,25.193,12.160,1.016,0.181,0.042,0.057,0.037
3,2025-12-31T18:56:42Z,909.765,2.896,34.814,23.749,12.582,1.044,0.287,0.041,0.042,0.028
4,2025-12-31T18:55:42Z,817.203,2.897,33.879,25.193,11.556,0.975,0.286,0.045,0.041,0.009


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:42,828.982,2.560,33.422,25.941,11.578,0.908,0.229,0.048,0.052,0.018
1,2025-12-31 18:58:42,875.709,2.768,33.418,25.930,11.323,0.940,0.248,0.054,0.089,0.018
2,2025-12-31 18:57:42,1005.216,2.897,34.116,25.193,12.160,1.016,0.181,0.042,0.057,0.037
3,2025-12-31 18:56:42,909.765,2.896,34.814,23.749,12.582,1.044,0.287,0.041,0.042,0.028
4,2025-12-31 18:55:42,817.203,2.897,33.879,25.193,11.556,0.975,0.286,0.045,0.041,0.009


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-05-01 09:00:00,756.44,9.51,43.03,2.15,4.35,0.49,0.17,0.04,0.05,0.04
1,2025-05-01 10:00:00,702.74,6.40,45.92,2.28,4.13,0.48,0.18,0.04,0.06,0.04
2,2025-05-01 11:00:00,662.67,7.86,50.85,2.44,4.04,0.44,0.17,0.04,0.06,0.04
3,2025-05-01 12:00:00,628.64,8.98,53.88,2.45,3.66,0.38,0.15,0.04,0.05,0.03
4,2025-05-01 13:00:00,646.78,9.16,55.28,2.45,3.98,0.38,0.15,0.04,0.05,0.03
...,...,...,...,...,...,...,...,...,...,...,...
5838,2025-12-31 14:00:00,716.60,31.61,32.42,1.95,11.85,0.81,0.22,0.04,0.04,0.02
5839,2025-12-31 15:00:00,728.30,32.36,31.70,1.84,10.74,0.80,0.22,0.03,0.04,0.02
5840,2025-12-31 16:00:00,729.69,32.82,29.93,2.35,11.77,0.95,0.27,0.04,0.04,0.02
5841,2025-12-31 17:00:00,784.93,34.06,25.45,2.43,10.93,1.15,0.30,0.05,0.05,0.03


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-05-01 09:00:00,756.44,9.51,43.03,2.15,4.35,0.49,0.17,0.04,0.05,0.04
2025-05-01 10:00:00,702.74,6.40,45.92,2.28,4.13,0.48,0.18,0.04,0.06,0.04
2025-05-01 11:00:00,662.67,7.86,50.85,2.44,4.04,0.44,0.17,0.04,0.06,0.04
2025-05-01 12:00:00,628.64,8.98,53.88,2.45,3.66,0.38,0.15,0.04,0.05,0.03
2025-05-01 13:00:00,646.78,9.16,55.28,2.45,3.98,0.38,0.15,0.04,0.05,0.03


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-05-01 09:00:00,0.501834,0.183910,0.488311,0.035096,0.043202,0.014219,0.009703,0.008969,0.014749,0.019512
2025-05-01 10:00:00,0.466209,0.123767,0.521108,0.037218,0.041017,0.013929,0.010274,0.008969,0.017699,0.019512
2025-05-01 11:00:00,0.439626,0.152002,0.577054,0.039830,0.040123,0.012768,0.009703,0.008969,0.017699,0.019512
2025-05-01 12:00:00,0.417050,0.173661,0.611439,0.039993,0.036349,0.011027,0.008562,0.008969,0.014749,0.014634
2025-05-01 13:00:00,0.429084,0.177142,0.627326,0.039993,0.039527,0.011027,0.008562,0.008969,0.014749,0.014634


In [11]:
data.to_csv('MOD-00684_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-05-01 09:00:00,0.490891,0.190792,0.480691,0.040887,0.109945,0.000000,0.000000,0.000225,0.012362,0.026900
2025-05-01 10:00:00,0.455052,0.131227,0.512639,0.038140,0.114386,0.000000,0.000000,0.000139,0.011151,0.024428
2025-05-01 11:00:00,0.429179,0.159424,0.568424,0.035871,0.114091,0.000000,0.000000,0.000193,0.009499,0.020619
2025-05-01 12:00:00,0.406271,0.181339,0.602623,0.033866,0.112378,0.000000,0.000000,0.000236,0.008196,0.017616
2025-05-01 13:00:00,0.418153,0.184843,0.618514,0.034863,0.115648,0.000000,0.000000,0.000240,0.008464,0.018201
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.475068,0.610301,0.370001,0.037446,0.103582,0.033372,0.016227,0.012300,0.021389,0.029765
2025-12-31 15:00:00,0.482243,0.625126,0.361423,0.038011,0.095815,0.029303,0.014248,0.010932,0.020241,0.029047
2025-12-31 16:00:00,0.483972,0.633779,0.341726,0.038060,0.103348,0.036023,0.017515,0.013240,0.022724,0.031415


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.015334,0.000000,0.135044,0.174751
1,0.009434,0.000000,0.150216,0.162179
2,0.013162,0.000000,0.164143,0.132849
3,0.016068,0.000000,0.172125,0.109753
4,0.016332,0.000000,0.176807,0.113606
...,...,...,...,...
5813,0.061334,0.016826,0.059588,0.085755
5814,0.062844,0.014775,0.055581,0.088513
5815,0.063668,0.018163,0.048746,0.089585
5816,0.065621,0.018178,0.031059,0.112640


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,4.289675,9.555707,2.893573,0.322612,0.000000,0.000000,0.000000,0.014694,0.067070,0.112078
Feature 2,0.519122,0.148088,0.000000,0.029103,3.146565,1.983349,0.964371,0.677451,0.696147,0.626022
Feature 3,0.805772,0.000000,3.230967,0.069980,0.496234,0.000000,0.000000,0.000000,0.000000,0.000000
Feature 4,1.810002,0.253315,0.000000,0.151587,0.245674,0.000000,0.000000,0.000000,0.064857,0.144102


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-05-01 09:00:00,0.015334,0.000000,0.135044,0.174751
2025-05-01 10:00:00,0.009434,0.000000,0.150216,0.162179
2025-05-01 11:00:00,0.013162,0.000000,0.164143,0.132849
2025-05-01 12:00:00,0.016068,0.000000,0.172125,0.109753
2025-05-01 13:00:00,0.016332,0.000000,0.176807,0.113606
...,...,...,...,...
2025-12-31 14:00:00,0.061334,0.016826,0.059588,0.085755
2025-12-31 15:00:00,0.062844,0.014775,0.055581,0.088513
2025-12-31 16:00:00,0.063668,0.018163,0.048746,0.089585


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,4.289675,9.555707,2.893573,0.322612,0.000000,0.000000,0.000000,0.014694,0.067070,0.112078
Factor 2,0.519122,0.148088,0.000000,0.029103,3.146565,1.983349,0.964371,0.677451,0.696147,0.626022
Factor 3,0.805772,0.000000,3.230967,0.069980,0.496234,0.000000,0.000000,0.000000,0.000000,0.000000
Factor 4,1.810002,0.253315,0.000000,0.151587,0.245674,0.000000,0.000000,0.000000,0.064857,0.144102


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.416061,0.031645,0.131897,0.420050,0.000348
no,0.394218,0.022351,0.144318,0.443209,-0.004095
no2,0.932155,0.009079,0.000000,0.059125,-0.000359
o3,0.346461,0.000000,0.652893,0.000000,0.000646
bin0,0.000000,0.589236,0.249531,0.175145,-0.013911
bin1,0.000000,1.075442,0.000000,0.000000,-0.075442
bin2,0.000000,1.058876,0.000000,0.000000,-0.058876
bin3,0.032619,0.945181,0.000000,0.000000,0.022200
bin4,0.098377,0.641756,0.000000,0.227621,0.032246
bin5,0.127380,0.447172,0.000000,0.391867,0.033581


In [20]:
results.to_csv('MOD-00684_factor_results.csv')
comp.to_csv('MOD-00684_factor_comp.csv')
res.to_csv('MOD-00684_factor_resid.csv')